In [1]:
%pip install -U \
    langchain \
    langchain-community \
    langchain-text-splitters \
    langchain-huggingface \
    pypdf \
    langchain-groq \
    python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
pip install faiss-cpu     # or faiss-gpu if you have NVIDIA + CUDA

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: Invalid requirement: '#': Expected package name at the start of dependency specifier
    #
    ^


In [1]:
# ─── Cell 1: Installs & Imports ──────────────────────────────────────────────

# Only run once (or when you change versions)
# !pip install -U langchain langchain-community langchain-text-splitters langchain-chroma langchain-huggingface pypdf langchain-groq python-dotenv sentence-transformers torch

import os
from dotenv import load_dotenv
load_dotenv()   # loads GROQ_API_KEY

# ─── Important LangChain imports ─────────────────────────────────────────────
from langchain_community.document_loaders import (
    PyPDFDirectoryLoader,
    DirectoryLoader,
    TextLoader
)
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

print("All core imports successful ✓")

c:\rag_chatbot\venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1
c:\rag_chatbot\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


All core imports successful ✓


In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

True

## Document Loader

In [3]:
class DocumentLoader:
    """Loads PDFs and TXT files from a folder"""
    def __init__(self, data_dir: str = "../data"):
        self.data_dir = data_dir

    def load(self):
        # Load all PDFs
        pdf_loader = PyPDFDirectoryLoader(self.data_dir)
        pdf_docs = pdf_loader.load()

        # Load all TXT files
        txt_loader = DirectoryLoader(
            self.data_dir,
            glob="**/*.txt",
            loader_cls=TextLoader,
            show_progress=True
        )
        txt_docs = txt_loader.load()

        print(f"Loaded {len(pdf_docs)} PDF pages + {len(txt_docs)} TXT files")
        return pdf_docs + txt_docs

## Chunking

In [4]:
class TextChunker:
    """Splits documents into small overlapping chunks"""
    def __init__(self, chunk_size: int = 500, chunk_overlap: int = 100):
        self.splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            length_function=len
        )

    def split(self, documents):
        chunks = self.splitter.split_documents(documents)
        print(f"Split into {len(chunks)} chunks")
        return chunks

## Vectore Store

In [ ]:


class VectorStore:
    def __init__(self, persist_dir: str = "faiss_index"):
        self.persist_dir = persist_dir
        self.embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
        self.vectorstore = None

    def build(self, chunks):
        self.vectorstore = FAISS.from_documents(
            documents=chunks,
            embedding=self.embeddings
        )
        self.vectorstore.save_local(self.persist_dir)          # <--- added
        print(f"FAISS index saved to {self.persist_dir}")
        return self.vectorstore

    def load(self):
        self.vectorstore = FAISS.load_local(
            self.persist_dir,
            embeddings=self.embeddings,
            allow_dangerous_deserialization=True   # needed in newer versions
        )
        print(f"Loaded FAISS index with {self.vectorstore.index.ntotal} vectors")
        return self.vectorstore

    def as_retriever(self, k: int = 5):
        return self.vectorstore.as_retriever(search_kwargs={"k": k})

## Rag Pipeline

In [6]:
class RAGChatbot:
    """Main chatbot class that ties everything together"""
    def __init__(self, data_dir: str = "../data"):
        self.data_dir = data_dir
        self.loader = DocumentLoader(data_dir)
        self.chunker = TextChunker()
        self.vector_store = VectorStore()
        self.llm = ChatGroq(
            model_name="llama-3.1-8b-instant",   # fast & cheap on Groq
            temperature=0.1,
            max_tokens=1024
        )
        self.retriever = None
        self.prompt = ChatPromptTemplate.from_template(
            """You are a helpful assistant.
Use ONLY the following context to answer the question.
If the answer is not in the context, say "I don't know based on the provided documents."

Context:
{context}

Question: {question}

Answer:"""
        )

    def build_index(self, force_rebuild: bool = False):
        """One-time indexing step"""
        if not force_rebuild and os.path.exists("chroma_db"):
            self.vector_store.load()
        else:
            docs = self.loader.load()
            chunks = self.chunker.split(docs)
            self.vector_store.build(chunks)

        self.retriever = self.vector_store.as_retriever(k=5)
        print("✅ RAG system is ready!")

    def answer(self, question: str):
        """Get answer for a single question"""
        if self.retriever is None:
            self.build_index()

        # Retrieve relevant chunks
        retrieved_docs = self.retriever.invoke(question)
        context = "\n\n".join([doc.page_content for doc in retrieved_docs])

        # Generate answer
        chain = self.prompt | self.llm | StrOutputParser()
        answer = chain.invoke({"context": context, "question": question})
        return answer

    def chat(self):
        """Interactive chatbot loop"""
        print("\n🤖 RAG Chatbot is live! (type 'exit' or 'quit' to stop)\n")
        while True:
            question = input("You: ").strip()
            if question.lower() in ["exit", "quit"]:
                print("Goodbye!")
                break
            if not question:
                continue

            answer = self.answer(question)
            print(f"Bot: {answer}\n")

In [ ]:
# ====================== RUN THE CHATBOT ======================
if __name__ == "__main__":
    chatbot = RAGChatbot(data_dir="../data")   # change path if needed
    chatbot.build_index(force_rebuild=False)   # set True only first time or when data changes
    chatbot.chat()

0it [00:00, ?it/s]


Loaded 78 PDF pages + 0 TXT files
Split into 342 chunks
FAISS index saved to faiss_index
✅ RAG system is ready!

🤖 RAG Chatbot is live! (type 'exit' or 'quit' to stop)

Bot: I don't know based on the provided documents.

Bot: I don't know based on the provided documents.

Bot: Chunking is the process of splitting large documents into smaller chunks using RecursiveCharacterTextSplitter.

Bot: The process involves the following steps:

1. Identifying the goal of a task (e.g., processing a return request)
2. Breaking down the task into smaller steps (e.g., asking the customer for order details, verifying the return policy eligibility)
3. Executing the task in parallel across multiple agents (e.g., multiple agents working together to complete the return process)
4. Evaluating and balancing various factors to optimize outcomes (e.g., evaluating urgency, return policies, and fees)
5. Verifying task completion against predefined success criteria or performance metrics
6. Providing feedback lo